# Binary Star Classification using Gaia DR3 Data

This notebook builds a simple machine learning pipeline to classify binary stars using Gaia DR3 data.

Steps of the pipeline:

1. Load a catalog of eclipsing binaries from the Kepler mission.
2. Cross-match the coordinates with Gaia DR3 sources.
3. Retrieve photometric and astrometric parameters from Gaia.
4. Construct a comparison sample of non-binary stars.
5. Train a Random Forest classifier.
6. Evaluate model performance using accuracy, sensitivity, and specificity.

In [46]:
from io import BytesIO
import requests
import pandas as pd
from astropy.io.votable import parse_single_table

from astroflow.enrich import enrich_df

## Load Kepler eclipsing binary catalog

The Kepler eclipsing binary catalog provides a list of confirmed binary systems with sky coordinates.
These coordinates will later be cross-matched with Gaia DR3 sources. We keep only the sky coordinates (RA, DEC) and the binary classification.
Missing coordinates are removed.


In [87]:
kepler = pd.read_csv("kepler.tsv", sep="\t", comment="#")
kepler = kepler[["_RA", "_DE", "binCl"]].copy()
kepler = kepler.rename(columns={"_RA": "ra", "_DE": "dec"})
kepler = kepler.dropna(subset=["ra", "dec"])
kepler["label"] = 1

kepler.head()

,ra,dec,binCl,label
0,deg,deg,,1
1,---------,---------,---,1
2,291.04407,+36.72927,D,1
3,291.25449,+36.74361,D,1
4,291.76196,+37.09864,D,1


In [88]:
kepler["binCl"].value_counts(dropna=False)
kepler.describe()
print(len(kepler))
kepler[["ra","dec"]].describe()

2179


,ra,dec
count,2179,2179
unique,2165,2167
top,287.16486,+43.23487
freq,3,3


Taking a sample of 300:

In [89]:
kepler_sample = kepler.sample(300, random_state=42).reset_index(drop=True)
kepler_sample.head()

,ra,dec,binCl,label
0,294.83679,+40.26560,D,1
1,287.67276,+42.05916,D,1
2,298.58252,+40.36473,SD,1
3,291.27692,+51.13571,D,1
4,288.36362,+42.24807,D,1


A token is neccessary for this pipeline:

In [90]:
import os
os.environ["GAIA_AIP_TOKEN"] = "7a7b9544f6f1c93a6e575df0b3595e1dcf95b7e4"

## Cross-match with Gaia DR3

Using the AstroFlow enrichment function, each Kepler coordinate is matched to the nearest Gaia DR3 source.
Astrometric and photometric parameters are then downloaded.

In [91]:
session = requests.Session()

kepler_enriched = enrich_df(
    session=session,
    df=kepler_sample,
    ra_col="ra",
    dec_col="dec",
    preset="basic",
    radius_arcsec=2.0,
)

kepler_enriched.head()

,ra_x,dec_x,binCl,label,__rowid__,source_id,matched_ra,matched_dec,separation_arcsec,ra_y,dec_y,phot_g_mean_mag,bp_rp,parallax,pmra,pmdec,ruwe
0,294.83679,+40.26560,D,1,1,2076497497990676992,294.836785,40.265567,0.119012,294.836785,40.265567,15.798594,1.029913,0.503484,-1.714692,-2.925062,1.045933
1,287.67276,+42.05916,D,1,2,2102404568915848192,287.672833,42.059049,0.444030,287.672833,42.059049,12.830001,0.804482,1.751565,9.421297,-22.964409,1.028766
2,298.58252,+40.36473,SD,1,3,2073584513744718208,298.582465,40.364705,0.175689,298.582465,40.364705,14.155263,1.133975,0.284588,-4.588597,-7.880794,1.167681
3,291.27692,+51.13571,D,1,4,2136191015048397184,291.276851,51.135632,0.321705,291.276851,51.135632,13.583560,0.797621,0.429367,-10.968462,-7.037315,24.404589
4,288.36362,+42.24807,D,1,5,2102489403109000576,288.363620,42.248025,0.161857,288.363620,42.248025,13.874940,0.663503,0.718764,-2.238801,-6.756515,1.005723


In [92]:
kepler_enriched = kepler_enriched.drop(columns=["ra_x","dec_x"])
kepler_enriched = kepler_enriched.rename(columns={"ra_y": "ra", "dec_y": "dec"})
kepler_enriched.head()

,binCl,label,__rowid__,source_id,matched_ra,matched_dec,separation_arcsec,ra,dec,phot_g_mean_mag,bp_rp,parallax,pmra,pmdec,ruwe
0,D,1,1,2076497497990676992,294.836785,40.265567,0.119012,294.836785,40.265567,15.798594,1.029913,0.503484,-1.714692,-2.925062,1.045933
1,D,1,2,2102404568915848192,287.672833,42.059049,0.444030,287.672833,42.059049,12.830001,0.804482,1.751565,9.421297,-22.964409,1.028766
2,SD,1,3,2073584513744718208,298.582465,40.364705,0.175689,298.582465,40.364705,14.155263,1.133975,0.284588,-4.588597,-7.880794,1.167681
3,D,1,4,2136191015048397184,291.276851,51.135632,0.321705,291.276851,51.135632,13.583560,0.797621,0.429367,-10.968462,-7.037315,24.404589
4,D,1,5,2102489403109000576,288.363620,42.248025,0.161857,288.363620,42.248025,13.874940,0.663503,0.718764,-2.238801,-6.756515,1.005723


## Verify cross-match quality

We examine the separation between the Kepler coordinates and the matched Gaia sources.
Small separations indicate successful matching.

In [93]:
kepler_enriched["separation_arcsec"].describe()


count    300.000000
mean       0.176545
std        0.156830
min        0.006930
25%        0.093967
50%        0.141988
75%        0.211518
max        1.665381
Name: separation_arcsec, dtype: float64

In [94]:
kepler_enriched.isna().sum()

binCl                0
label                0
__rowid__            0
source_id            0
matched_ra           0
matched_dec          0
separation_arcsec    0
ra                   0
dec                  0
phot_g_mean_mag      0
bp_rp                0
parallax             4
pmra                 4
pmdec                4
ruwe                 4
dtype: int64

In [95]:
kepler_enriched = kepler_enriched.dropna(
    subset=["parallax","pmra","pmdec"]
)
len(kepler_enriched)

296

## Construct a non-binary comparison sample

A random sample of Gaia DR3 stars is downloaded to represent non-binary systems.
This creates a balanced dataset for classification.

In [96]:
query = """
SELECT TOP 300
    source_id,
    ra,
    dec,
    phot_g_mean_mag,
    bp_rp,
    parallax,
    pmra,
    pmdec,
    ruwe
FROM gaiadr3.gaia_source
WHERE
    parallax IS NOT NULL
    AND pmra IS NOT NULL
    AND pmdec IS NOT NULL
ORDER BY random_index
"""

url = "https://gaia.aip.de/tap/sync"

response = requests.post(
    url,
    data={
        "REQUEST": "doQuery",
        "LANG": "ADQL",
        "QUERY": query,
    },
)

table = parse_single_table(BytesIO(response.content)).to_table()
non_binary = table.to_pandas()

non_binary["label"] = 0
non_binary.head()

,datalinkID,ra,dec,phot_g_mean_mag,bp_rp,parallax,pmra,pmdec,ruwe,label
0,4267180339403392768,286.716913,0.276195,15.244129,1.522128,1.084924,2.549028,-4.075544,1.033783,0
1,5252403815119316480,152.625068,-64.267699,20.906347,0.702095,0.854356,-2.606806,4.414332,0.912624,0
2,1937745177867542656,348.434982,43.940704,20.531225,1.649284,1.042008,-1.729704,-3.353289,1.051422,0
3,5971301282285218048,252.729717,-37.917215,20.145899,2.006773,0.587660,-3.566062,-4.132508,1.083541,0
4,5953201637243132416,260.678787,-44.788772,19.787357,1.573410,-0.293376,0.209898,-3.433852,1.251593,0


In [97]:
print(non_binary.shape)
non_binary.isna().sum()

(300, 10)


datalinkID          0
ra                  0
dec                 0
phot_g_mean_mag     0
bp_rp              27
parallax            0
pmra                0
pmdec               0
ruwe                0
label               0
dtype: int64

In [98]:
non_binary = non_binary.dropna(subset=["bp_rp"])
print(non_binary.shape)

(273, 10)


In [99]:
non_binary = non_binary.drop(columns=["datalinkID"], errors="ignore")

## Feature selection

The following Gaia parameters are used as features:

- phot_g_mean_mag (G-band magnitude)
- bp_rp (color index)
- parallax (distance indicator)
- pmra, pmdec (proper motion components)
- ruwe (astrometric goodness-of-fit)

In [100]:
features = [
    "phot_g_mean_mag",
    "bp_rp",
    "parallax",
    "pmra",
    "pmdec",
    "ruwe"
]

binary_df = kepler_enriched[features + ["label"]].copy()
non_binary_df = non_binary[features + ["label"]].copy()

dataset = pd.concat([binary_df, non_binary_df], ignore_index=True)
dataset = dataset.dropna()

dataset["label"].value_counts()

label
1    296
0    273
Name: count, dtype: int64

## Train Random Forest classifier

The dataset is split into training and testing subsets.
A Random Forest classifier is trained to distinguish binary and non-binary stars.
Model performance is evaluated using:

- Accuracy
- Sensitivity (true positive rate)
- Specificity (true negative rate)
- Confusion matrix

In [101]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report

X = dataset[features]
y = dataset["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

rf = RandomForestClassifier(n_estimators=300, random_state=42)
rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)

cm = confusion_matrix(y_test, y_pred)
TN, FP, FN, TP = cm.ravel()

accuracy = (TP + TN) / (TP + TN + FP + FN)
sensitivity = TP / (TP + FN)
specificity = TN / (TN + FP)
precision = TP / (TP + FP)

print("Confusion matrix:\n", cm)
print()
print("Accuracy:", round(accuracy, 3))
print("Sensitivity (Recall, TPR):", round(sensitivity, 3))
print("Specificity (TNR):", round(specificity, 3))
print("Precision:", round(precision, 3))
print()
print(classification_report(y_test, y_pred))

importance = pd.Series(rf.feature_importances_, index=features).sort_values(ascending=False)
print("Feature importance:")
print(importance)

Confusion matrix:
 [[51  4]
 [ 1 58]]

Accuracy: 0.956
Sensitivity (Recall, TPR): 0.983
Specificity (TNR): 0.927
Precision: 0.935

              precision    recall  f1-score   support

           0       0.98      0.93      0.95        55
           1       0.94      0.98      0.96        59

    accuracy                           0.96       114
   macro avg       0.96      0.96      0.96       114
weighted avg       0.96      0.96      0.96       114

Feature importance:
phot_g_mean_mag    0.591474
bp_rp              0.228161
parallax           0.111805
ruwe               0.031341
pmra               0.020433
pmdec              0.016786
dtype: float64
